In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("pandas version:", pd.__version__)

pandas version: 2.2.3


In [ ]:

# Load dataset (Level 1 output first, raw fallback second)

CANDIDATE_PATHS = [
    "level1/output/trains_cleaned.csv",
    "trains_cleaned.csv",
    "Railway_info.csv",
    "data/raw/Railway_info.csv",
    "trains.csv",
    "data/raw/trains.csv",
]

def load_dataset():
    """Return (dataframe, source_path). Falls back to a Colab upload widget."""
    for path in CANDIDATE_PATHS:
        if os.path.exists(path):
            print(f"Loading dataset from: {path}")
            return pd.read_csv(path), path
    try:  # Google Colab upload fallback
        from google.colab import files
        print("Dataset not found in this session — please upload your CSV:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        return pd.read_csv(name), name
    except ImportError:
        raise FileNotFoundError(
            "Could not locate the dataset CSV. Upload it (Colab: folder icon) and re-run."
        )

df, source_path = load_dataset()

# Defensive column resolution (robust to header naming)
def find_column(frame, *patterns):
    """First column whose lower-cased name contains any of the patterns."""
    for pattern in patterns:
        for col in frame.columns:
            if pattern in col.lower():
                return col
    return None

train_col  = find_column(df, "train no", "train_no", "train id", "train code", "train") or df.columns[0]
source_col = find_column(df, "source", "origin") or find_column(df, "from")
dest_col   = find_column(df, "destination") or find_column(df, "to")
day_col    = find_column(df, "day") or "days"

# If we fell back to RAW data, apply Level 1 standardization inline
if "cleaned" not in os.path.basename(source_path).lower():
    print("Raw data loaded — applying Level 1 standardization inline.")
    def _std(v):
        return v if pd.isna(v) else " ".join(str(v).split()).upper()
    for col in {source_col, dest_col, find_column(df, "train name")}:
        if col:
            df[col] = df[col].apply(_std)

# Normalize day names for reliable filtering (' saturday' / 'SATURDAY' -> 'Saturday')
# and repair corrupted tokens found in the raw feed (e.g. 'Fridayd' -> 'Friday')
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def normalize_day(value):
    v = str(value).strip().capitalize()
    for day in DAYS:
        if v.startswith(day):
            return day
    return v

_clean = df[day_col].fillna("UNKNOWN").astype(str).str.strip().str.capitalize()
n_dirty = int((~_clean.isin(DAYS)).sum())
df[day_col] = _clean.apply(normalize_day)
print(f"Repaired {n_dirty:,} corrupted day value(s) (e.g. 'Fridayd' -> 'Friday').")

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head(10)

Loading dataset from: Railway_info.csv
Raw data loaded — applying Level 1 standardization inline.
Repaired 1,153 corrupted day value(s) (e.g. 'Fridayd' -> 'Friday').
Rows: 11,113 | Columns: 5


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
0,107,SWV-MAO-VLNK,SAWANTWADI ROAD,MADGOAN JN.,Saturday
1,108,VLNK-MAO-SWV,MADGOAN JN.,SAWANTWADI ROAD,Friday
2,128,MAO-KOP SPEC,MADGOAN JN.,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Friday
3,290,PALACE ON WH,DELHI-SAFDAR JANG,DELHI-SAFDAR JANG,Wednesday
4,401,BSB BHARATDA,AURANGABAD,VARANASI JN.,Saturday
5,421,LKO-SVDK FTR,LUCKNOW JN.,SHRI MATA VAISHNO DEVI KATRA,Tuesday
6,422,SVDK-LKO FTR,SHRI MATA VAISHNO DEVI KATRA,LUCKNOW JN.,Monday
7,477,FTR TRAIN NO,SIRSA,SIRSA,Sunday
8,502,RJPB-UMB FTR,RAJENDRANAGAR TERMINAL,AMBALA CANTT JN,Monday
9,504,PNBE-BTI FTR,PATNA JN.,BATHINDA JN,Wednesday


In [ ]:

# Task 2.1 | Filter trains operating on a specific day

TARGET_DAY = "Saturday"   # <- change to any day: Monday ... Sunday

print("Services per day in the dataset:")
print(df[day_col].value_counts())

day_df = df[df[day_col] == TARGET_DAY].copy()
print(f"\n🚆 Trains operating on {TARGET_DAY}: {len(day_df):,}")
day_df.head(10)

Services per day in the dataset:
days
Friday       1649
Tuesday      1628
Wednesday    1612
Sunday       1602
Saturday     1593
Thursday     1526
Monday       1503
Name: count, dtype: int64

🚆 Trains operating on Saturday: 1,593


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
0,107,SWV-MAO-VLNK,SAWANTWADI ROAD,MADGOAN JN.,Saturday
4,401,BSB BHARATDA,AURANGABAD,VARANASI JN.,Saturday
21,1196,NGP-KRMI SPL,NAGPUR JN.(CR),KARMALI,Saturday
28,1706,JBP-BDTS SF,JABALPUR,BANDRA TERMINUS,Saturday
45,2834,SRC-RJT SF A,SANTRAGACHI JN.,RAJKOT,Saturday
54,3305,DHN-KUSUNDA,DHANBAD JN.,KUSUNDA,Saturday
59,3502,ANVT-JSME BI,ANAND VIHAR TERMINAL,JASIDIH JN.,Saturday
77,4802,MKN PBC PASS,MAKRANA JN.,PARVATSAR CITY,Saturday
91,5066,LJN-CPR-EXP,LUCKNOW JN.,CHHAPRA JN.,Saturday
95,5306,FBD- CPA EXP,FARRUKHABAD JN,KANPUR ANWARGANJ,Saturday


In [ ]:

# Task 2.1 | New dataframe: trains starting from a specific station

# Default = busiest source station; set any UPPER-CASE station name to explore others
SOURCE_STATION = df[source_col].value_counts().idxmax()

station_df = df[df[source_col] == SOURCE_STATION].copy()
print(f"🚉 Trains starting from {SOURCE_STATION}: {len(station_df):,}")
station_df.head(10)

🚉 Trains starting from CST-MUMBAI: 513


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days
14,1011,CSMT-NGP SF,CST-MUMBAI,NAGPUR JN.(CR),Thursday
31,2025,CSMT-KRMI SF,CST-MUMBAI,KARMALI,Tuesday
273,10103,MANDOVI EXPR,CST-MUMBAI,MADGOAN JN.,Thursday
275,10111,KONKAN KANYA,CST-MUMBAI,MADGOAN JN.,Friday
285,11007,DECCAN EXPRE,CST-MUMBAI,PUNE JN.,Wednesday
287,11009,SINHAGAD EXP,CST-MUMBAI,PUNE JN.,Thursday
297,11019,KONARK EXPRE,CST-MUMBAI,BHUBANESWAR,Friday
301,11023,SAHYADRI EXP,CST-MUMBAI,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Wednesday
305,11027,CSMT-MAS MAI,CST-MUMBAI,CHENNAI CENTRAL,Sunday
307,11029,KOYNA EXPRES,CST-MUMBAI,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Tuesday


In [ ]:

# Task 2.2 | Trains originating per source station

trains_per_source = (
    df.groupby(source_col)[train_col]
      .count()
      .rename("train_count")
      .sort_values(ascending=False)
)

print("Trains originating per source station (top 10):")
print(trains_per_source.head(10))
print(f"\nStations covered: {len(trains_per_source):,}")

Trains originating per source station (top 10):
Source_Station_Name
CST-MUMBAI       513
SEALDAH          372
CHENNAI BEACH    339
HOWRAH JN.       338
KALYAN JN        285
THANE            186
PANVEL           141
TAMBARAM         140
MOOR MARKET      135
VELACHEERY       115
Name: train_count, dtype: int64

Stations covered: 921


In [ ]:

# Task 2.2 | Average trains per day, per source station

# Count trains per (station, day) pair, then average over the days each station runs
per_day = df.groupby([source_col, day_col])[train_col].count()

avg_trains_per_day = (
    per_day.groupby(level=0)
           .mean()
           .round(2)
           .rename("avg_trains_per_day")
           .sort_values(ascending=False)
)

print("Average trains per operating day, per source station (top 10):")
print(avg_trains_per_day.head(10))

Average trains per operating day, per source station (top 10):
Source_Station_Name
CST-MUMBAI       73.29
SEALDAH          53.14
CHENNAI BEACH    48.43
HOWRAH JN.       48.29
KALYAN JN        40.71
THANE            26.57
PANVEL           20.14
TAMBARAM         20.00
MOOR MARKET      19.29
VELACHEERY       16.43
Name: avg_trains_per_day, dtype: float64


In [ ]:

# Task 2.3 | Categorize operating days: Weekday vs Weekend

def categorize_day(day):
    if day in {"Saturday", "Sunday"}:
        return "Weekend"
    if day in {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday"}:
        return "Weekday"
    return "Unknown"   # never silently mislabel unexpected values

df["Day_Category"] = df[day_col].apply(categorize_day)

print("Day_Category distribution:")
print(df["Day_Category"].value_counts())
df.head(10)

Day_Category distribution:
Day_Category
Weekday    7918
Weekend    3195
Name: count, dtype: int64


,Train_No,Train_Name,Source_Station_Name,Destination_Station_Name,days,Day_Category
0,107,SWV-MAO-VLNK,SAWANTWADI ROAD,MADGOAN JN.,Saturday,Weekend
1,108,VLNK-MAO-SWV,MADGOAN JN.,SAWANTWADI ROAD,Friday,Weekday
2,128,MAO-KOP SPEC,MADGOAN JN.,CHHATRAPATI SHAHU MAHARAJ TERMINUS,Friday,Weekday
3,290,PALACE ON WH,DELHI-SAFDAR JANG,DELHI-SAFDAR JANG,Wednesday,Weekday
4,401,BSB BHARATDA,AURANGABAD,VARANASI JN.,Saturday,Weekend
5,421,LKO-SVDK FTR,LUCKNOW JN.,SHRI MATA VAISHNO DEVI KATRA,Tuesday,Weekday
6,422,SVDK-LKO FTR,SHRI MATA VAISHNO DEVI KATRA,LUCKNOW JN.,Monday,Weekday
7,477,FTR TRAIN NO,SIRSA,SIRSA,Sunday,Weekend
8,502,RJPB-UMB FTR,RAJENDRANAGAR TERMINAL,AMBALA CANTT JN,Monday,Weekday
9,504,PNBE-BTI FTR,PATNA JN.,BATHINDA JN,Wednesday,Weekday


In [ ]:

# Export Level 2 outputs

os.makedirs("level2/output", exist_ok=True)

df.to_csv("level2/output/trains_enriched.csv", index=False)
day_df.to_csv(f"level2/output/trains_{TARGET_DAY.lower()}.csv", index=False)
station_file = f"level2/output/trains_from_{SOURCE_STATION.lower().replace(' ', '_')}.csv"
station_df.to_csv(station_file, index=False)

print(" Saved Level 2 outputs:")
print("   level2/output/trains_enriched.csv   (full enriched dataset)")
print(f"   level2/output/trains_{TARGET_DAY.lower()}.csv   ({TARGET_DAY} filter)")
print(f"   {station_file}   ({SOURCE_STATION} filter)")

try:  # auto-download when running in Colab
    from google.colab import files
    files.download("level2/output/trains_enriched.csv")
except ImportError:
    pass

💾 Saved Level 2 outputs:
   level2/output/trains_enriched.csv   (full enriched dataset)
   level2/output/trains_saturday.csv   (Saturday filter)
   level2/output/trains_from_cst-mumbai.csv   (CST-MUMBAI filter)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>